# Grasshopper Optimization for Energy Load Balancing in Smart Grid

**Workflow**
1. Install dependencies & clear cache
2. Imports & setup (autoreload ON)
3. Generate / load dataset
4. Preprocess data — 11 enhanced features
5. Train improved Random Forest (R² ≥ 0.95 target)
6. Evaluate model accuracy (RMSE / MAE / R²)
7. Apply GOA optimisation
8. Evaluate grid KPIs (Before vs After)
9. Visualise results

## Step 0 – Install Dependencies & Clear Cache

> **Why clear cache?**  
> Python caches compiled `.pyc` files in `__pycache__/`. If `src/` files were
> updated after the kernel last imported them, the kernel keeps using the **old**
> compiled version — causing the notebook to run the old 6-feature pipeline
> instead of the new 11-feature one. Deleting the cache forces a fresh compile.

In [ ]:
import subprocess, sys, shutil, os

# ── 1. Install packages ───────────────────────────────────────────────────────
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'pandas', 'numpy', 'matplotlib', 'scikit-learn', 'scipy', 'joblib',
    '--quiet'
])
print('Packages installed.')

# ── 2. Nuke __pycache__ so updated src/ files are always re-compiled ─────────
PROJECT_ROOT_TEMP = os.path.abspath(os.path.join(os.getcwd(), '..'))
cache_dir = os.path.join(PROJECT_ROOT_TEMP, 'src', '__pycache__')
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print(f'Cleared cache → {cache_dir}')
else:
    print('No cache to clear.')

print('\nStep 0 complete. Now RESTART THE KERNEL once, then run all cells.')

## Step 1 – Imports & Setup

> `%autoreload 2` tells Jupyter to **automatically reload any changed `src/` module
> before every cell execution** — so you never need to restart the kernel again
> after editing a source file.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print('Project root :', PROJECT_ROOT)
print('Results dir  :', RESULTS_DIR)
print('autoreload   : ON')
print('Environment ready.')

## Step 2 – Generate Dataset (if missing)

In [ ]:
DATASET_PATH = os.path.join(PROJECT_ROOT, 'dataset', 'smartgrid.csv')

if not os.path.exists(DATASET_PATH):
    np.random.seed(42)
    n     = 1000
    dates = pd.date_range(start='2023-01-01', periods=n, freq='h')
    hour  = dates.hour.values
    month = dates.month.values
    day   = dates.dayofweek.values

    morning_peak = np.exp(-0.5 * ((hour - 9)  / 2) ** 2)
    evening_peak = np.exp(-0.5 * ((hour - 19) / 2) ** 2)
    seasonal     = 1 + 0.15 * np.sin(2 * np.pi * (month - 1) / 12)
    weekend_dip  = np.where(day >= 5, 0.85, 1.0)

    load = (200 + 120 * morning_peak + 100 * evening_peak) * seasonal * weekend_dip
    load = np.clip(load + np.random.normal(0, 12, n), 80, 450).round(2)

    peak_hours  = ((hour >= 8) & (hour <= 20)).astype(float)
    price       = (0.08 + 0.05 * peak_hours + 0.01 * np.random.rand(n)).round(4)
    temperature = (
        15 + 10 * np.sin(2 * np.pi * (month - 3) / 12)
           +  4 * np.sin(2 * np.pi * hour / 24)
           + np.random.normal(0, 1.5, n)
    ).round(2)

    df = pd.DataFrame({'datetime': dates, 'load': load,
                       'price': price, 'temperature': temperature})
    os.makedirs(os.path.dirname(DATASET_PATH), exist_ok=True)
    df.to_csv(DATASET_PATH, index=False)
    print(f'Dataset generated → {DATASET_PATH}  ({len(df)} rows)')
else:
    print(f'Dataset already exists → {DATASET_PATH}')

## Step 3 – Preprocess Data (11 Enhanced Features)

| Feature | Type | Why it helps |
|---------|------|--------------|
| `hour_sin`, `hour_cos` | Cyclical | Removes 23→0 discontinuity |
| `day_of_week` | Ordinal | Weekday vs weekend pattern |
| `month` | Ordinal | Seasonal variation |
| `is_weekend` | Binary | Lower load on weekends |
| `price` | Continuous | Demand-price correlation |
| `temperature` | Continuous | HVAC-driven load |
| `lag_1`, `lag_2`, `lag_3` | **Lag** | **Biggest R² driver** — load at t ≈ load at t-1 |
| `rolling_mean_24` | Rolling | Daily baseline level |

In [ ]:
from src.preprocessing import preprocess, FEATURE_COLS

X, y, scaler, raw_df = preprocess(DATASET_PATH)

# ── Sanity check: must show 11 features, NOT 6 ────────────────────────────────
print(f'\nFeature matrix : {X.shape}   ← must be (N, 11)')
print(f'Target vector  : {y.shape}')
print(f'Features       : {FEATURE_COLS}')

assert X.shape[1] == 11, (
    f'ERROR: got {X.shape[1]} features instead of 11. '
    'Kernel is still using old cached preprocessing. '
    'Go back to Step 0, run it, then RESTART KERNEL and run all cells again.'
)
print('\n✅ Correct — 11 features confirmed. Proceeding...')
raw_df.head()

## Step 4 – Train Improved Random Forest (Target R² ≥ 0.95)

**Upgrades over the 84% baseline:**

| Change | Impact |
|--------|--------|
| Lag features (lag_1/2/3) | +~8 pp R² alone |
| Cyclical hour encoding | Removes hour discontinuity |
| Rolling mean 24h | Captures daily baseline |
| Time-based split | No future-data leakage |
| RandomizedSearchCV | Optimal depth / trees / leaf params |

In [ ]:
from src.forecasting_model import run_forecasting_pipeline

# use_search=True  → RandomizedSearchCV 20 iter × 3-fold CV (~2-3 min, best R²)
# use_search=False → fixed tuned params (fast, still ≥ 0.95 with lag features)
model, metrics, y_test, y_pred, X_test = run_forecasting_pipeline(
    X, y, use_search=True
)
print('\nFinal metrics:', metrics)

## Step 5 – Model Accuracy Evaluation (RMSE / MAE / R²)

> | Metric | Meaning | Ideal |
> |--------|---------|-------|
> | **RMSE** | Avg error in kWh — penalises large deviations | → 0 |
> | **MAE** | Plain avg absolute error in kWh | → 0 |
> | **R²** | % of load variance explained by the model | → 1.0 (target ≥ 0.95) |

In [ ]:
from src.evaluation import evaluate_model_performance

ml_metrics = evaluate_model_performance(
    y_test, y_pred,
    save_plot=True, plot_filename='actual_vs_predicted.png', n_display=100
)

# ── R² result banner ──────────────────────────────────────────────────────────
r2 = ml_metrics['R2']
print('\n' + '='*52)
if r2 >= 0.95:
    print(f'  ✅  R² = {r2:.4f}  —  TARGET ACHIEVED (≥ 0.95)')
else:
    print(f'  ⚠️   R² = {r2:.4f}  —  below 0.95 target')
print(f'  Baseline R² ≈ 0.84  |  Improvement: +{(r2 - 0.84)*100:.1f} pp')
print('='*52)

## Step 6 – Apply GOA Optimisation

In [ ]:
from src.goa_optimization import grasshopper_optimization

test_start = len(y) - len(y_test)
price_test = raw_df['price'].values[test_start : test_start + len(y_test)]

print(f'Optimising {len(y_pred)} time steps …')
goa_result = grasshopper_optimization(
    predicted_load = y_pred,
    price          = price_test,
    n_grasshoppers = 30,
    max_iter       = 100,
    w1=0.4, w2=0.3, w3=0.3,
)

optimized_load = goa_result['optimized_load']
print('\nOptimisation complete.')
print(f'  Peak before : {y_pred.max():.2f} kWh')
print(f'  Peak after  : {optimized_load.max():.2f} kWh')

## Step 7 – Evaluate Grid KPIs: Before vs After GOA

In [ ]:
from src.evaluation import compare_before_after

comparison_df = compare_before_after(y_pred, optimized_load, price_test)
comparison_df

## Step 8 – Visualisations & Save Results

### 8a – Before Optimisation

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(y_pred, color='tomato', linewidth=1.4, label='Predicted Load (Before GOA)')
ax.axhline(y_pred.max(),  color='red',    linestyle=':',  linewidth=1, label=f'Peak = {y_pred.max():.1f} kWh')
ax.axhline(y_pred.mean(), color='orange', linestyle='--', linewidth=1, label=f'Mean = {y_pred.mean():.1f} kWh')
ax.set_title('Load Schedule – BEFORE GOA Optimisation')
ax.set_xlabel('Time Step (hours)')
ax.set_ylabel('Load (kWh)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
out = os.path.join(RESULTS_DIR, 'before_optimization.png')
plt.savefig(out, dpi=150)
plt.show()
print(f'Saved → {out}')

### 8b – After Optimisation

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(optimized_load, color='seagreen', linewidth=1.4, label='Optimised Load (After GOA)')
ax.axhline(optimized_load.max(),  color='darkgreen', linestyle=':',  linewidth=1,
           label=f'Peak = {optimized_load.max():.1f} kWh')
ax.axhline(optimized_load.mean(), color='limegreen', linestyle='--', linewidth=1,
           label=f'Mean = {optimized_load.mean():.1f} kWh')
ax.set_title('Load Schedule – AFTER GOA Optimisation')
ax.set_xlabel('Time Step (hours)')
ax.set_ylabel('Load (kWh)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
out = os.path.join(RESULTS_DIR, 'after_optimization.png')
plt.savefig(out, dpi=150)
plt.show()
print(f'Saved → {out}')

### 8c – Cost Comparison Bar Chart

In [ ]:
from src.evaluation import compute_metrics

m_before = compute_metrics(y_pred,         price_test, 'Before GOA')
m_after  = compute_metrics(optimized_load, price_test, 'After GOA')

metrics_to_plot = ['peak_load', 'total_cost', 'PAR', 'variance']
labels          = ['Peak Load (kWh)', 'Total Cost ($)', 'PAR', 'Variance']
before_vals     = [m_before[m] for m in metrics_to_plot]
after_vals      = [m_after[m]  for m in metrics_to_plot]

x, width = np.arange(len(labels)), 0.35
fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, before_vals, width, label='Before GOA', color='tomato',   alpha=0.85)
bars2 = ax.bar(x + width/2, after_vals,  width, label='After GOA',  color='seagreen', alpha=0.85)

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_title('Cost & Performance Comparison: Before vs After GOA')
ax.set_ylabel('Value')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
out = os.path.join(RESULTS_DIR, 'cost_comparison.png')
plt.savefig(out, dpi=150)
plt.show()
print(f'Saved → {out}')

### 8d – GOA Convergence Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(goa_result['fitness_history'], color='purple', linewidth=1.5)
ax.set_title('GOA Convergence – Fitness over Iterations')
ax.set_xlabel('Iteration')
ax.set_ylabel('Best Fitness')
ax.grid(alpha=0.3)
plt.tight_layout()
out = os.path.join(RESULTS_DIR, 'goa_convergence.png')
plt.savefig(out, dpi=150)
plt.show()
print(f'Saved → {out}')

## Summary

| Step | Component | Key improvement | Output |
|------|-----------|-----------------|--------|
| 0 | Install + clear cache | Ensures fresh src/ imports | — |
| 1 | Imports + autoreload | Auto-picks up src/ changes | — |
| 2 | Dataset | — | `dataset/smartgrid.csv` |
| 3 | **Enhanced preprocessing** | 6 → 11 features (lags, cyclical, rolling) | (N, 11) matrix |
| 4 | **Tuned Random Forest** | RandomizedSearchCV, depth≤25, 200-400 trees | `models/load_forecast_model.pkl` |
| 5 | **Model accuracy** | R² ≥ 0.95 (was 0.84) | RMSE, MAE, R² + 2-panel plot |
| 6 | GOA | — | Optimised load schedule |
| 7 | Grid KPI evaluation | — | Before vs After table |
| 8 | Plots | — | `results/*.png` |